# OpenWEC — Le Mans 2026 Analysis

This notebook demonstrates the OpenWEC Python SDK using the 2026 24 Hours of Le Mans as a case study.

**What we'll cover:**
- Loading race results and lap data
- Analyzing HYPERCAR stint strategies
- Comparing pace across manufacturers
- Visualizing gap to leader and lap evolution

**Requirements:**
```bash
pip install openwec[plotting]
```

**API key:** Public endpoints (results) work without a key. Laps and analytics require a free key — [request one here](https://openwec.com/api-keys).

## 1. Setup

In [ ]:
import openwec
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Load API key from .env file
env_file = Path(".env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.startswith("API_KEYS="):
            key = line.split("=", 1)[1].strip().split(",")[0]
            os.environ.setdefault("OPENWEC_API_KEY", key)

openwec.configure(
    base_url="https://api.openwec.com/api/v1",
    api_key=os.environ.get("API_KEYS", "")
)

print(f"openwec {openwec.__version__}")

## 2. Load the session

In [ ]:
session = openwec.Session("WEC", 2026, "Le Mans", "Race")
print(session)

Expected output:
```
Session(WEC 2026 LE MANS — Race, id=6556)
```

## 3. Race results

Results are public — no API key required.

In [ ]:
results = session.results()
print(f"{len(results)} cars classified")
results.head(10)

In [ ]:
# HYPERCAR top 5
hypercar = results[results["car_class"] == "HYPERCAR"].copy()
hypercar[["position", "car_number", "team", "vehicle", "laps_completed", "gap_to_first_s", "drivers"]].head(5)

In [ ]:
# Winner info
winner = results[results["position"] == 1].iloc[0]
print(f"Winner: #{winner['car_number']} {winner['team']}")
print(f"Vehicle: {winner['vehicle']}")
print(f"Drivers: {winner['drivers']}")
print(f"Laps: {winner['laps_completed']}")
print(f"Fastest lap: {winner['fl_time_s']:.3f}s (lap {winner['fl_lap_number']})")

## 4. Pace analysis

Average green-flag pace per car, sorted fastest first. Requires API key.

In [ ]:
pace = session.pace(car_class="HYPERCAR")
pace[["car_number", "team", "avg_pace_s", "best_lap_s", "consistency_s", "pit_stops"]].head(10)

In [ ]:
# Average pace by manufacturer
# Merge with results to get vehicle info
pace_enriched = pace.merge(
    results[["car_number", "vehicle"]],
    on="car_number",
    how="left"
)

# Extract manufacturer from vehicle name
pace_enriched["manufacturer"] = pace_enriched["vehicle"].str.split().str[0]

manufacturer_pace = (
    pace_enriched.groupby("manufacturer")["avg_pace_s"]
    .mean()
    .sort_values()
    .reset_index()
)

print("Average HYPERCAR pace by manufacturer:")
for _, row in manufacturer_pace.iterrows():
    m = int(row['avg_pace_s'] // 60)
    s = row['avg_pace_s'] % 60
    print(f"  {row['manufacturer']:12s} {m}:{s:06.3f}")

## 5. Stint analysis

Stint boundaries, baseline pace, degradation rate, and consistency per stint per car.

In [ ]:
stints = session.stints(car_class="HYPERCAR")
print(f"{len(stints)} stints total across HYPERCAR class")
stints.head(10)

In [ ]:
# Pit stop count per car
pit_counts = (
    stints.groupby("car_number")["stint_number"]
    .max()
    .subtract(1)  # stints - 1 = pit stops
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"stint_number": "pit_stops"})
)

print("Pit stops per HYPERCAR:")
print(pit_counts.to_string(index=False))

In [ ]:
# Average degradation per car (positive = getting slower)
degradation = (
    stints[stints["degradation_s_per_lap"].notna()]
    .groupby(["car_number", "team"])["degradation_s_per_lap"]
    .mean()
    .sort_values()
    .reset_index()
)

print("Average degradation (s/lap) — negative = improving:")
print(degradation.to_string(index=False))

## 6. Lap-by-lap analysis

Deep dive into a single car — the race winner.

In [ ]:
# Get winner car number
winner_car = results[results["position"] == 1]["car_number"].iloc[0]
print(f"Loading laps for car #{winner_car}...")

laps = session.laps(car=str(winner_car))
print(f"{len(laps)} laps loaded")
laps[["lap_number", "lap_time_s", "s1_s", "s2_s", "s3_s", "crossing_finish_in_pit", "flag_at_fl"]].head(10)

In [ ]:
# Green flag laps only
green = laps[
    (laps["flag_at_fl"] == "GF") &
    (laps["crossing_finish_in_pit"] == False) &
    (laps["lap_time_s"].notna()) &
    (laps["lap_time_s"] < 600)
].copy()

print(f"Green flag laps: {len(green)} / {len(laps)} total")
print(f"Best lap:   {green['lap_time_s'].min():.3f}s")
print(f"Mean pace:  {green['lap_time_s'].mean():.3f}s")
print(f"Std dev:    {green['lap_time_s'].std():.3f}s")

## 7. Visualizations

Requires `pip install openwec[plotting]`

In [ ]:
# Lap evolution — winner car
fig = session.plot_lap_evolution(car=str(winner_car))
fig.suptitle(f"Le Mans 2026 — Car #{winner_car} Lap Evolution", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Stint chart — HYPERCAR class strategy overview
fig = session.plot_stint_chart(car_class="HYPERCAR")
fig.suptitle("Le Mans 2026 — HYPERCAR Stint Strategy", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gap to leader — first 60 laps
fig = session.plot_gap_to_leader(car_class="HYPERCAR", max_laps=60)
fig.suptitle("Le Mans 2026 — HYPERCAR Gap to Leader (Laps 1-60)", y=1.02)
plt.tight_layout()
plt.show()

## 8. Pit window estimator

In [ ]:
# Optimal pit window for the winner
pit_window = session.pit_window(car=str(winner_car))
pit_window[["stint_number", "start_lap", "end_lap", "tyre_age_laps",
            "baseline_pace_s", "degradation_s_per_lap", 
            "early_lap_abs", "ideal_lap_abs", "late_lap_abs",
            "recommendation"]].head(10)

## 9. Exploring other sessions

The same API works for any series and session.

In [ ]:
# ELMS 2025 — Spa
elms_spa = openwec.Session("ELMS", 2025, "Spa", "Race")
print(elms_spa)

elms_results = elms_spa.results()
elms_results[["position", "car_number", "car_class", "team", "laps_completed"]].head(10)

In [ ]:
# IMSA 2026 — Daytona
imsa = openwec.Session("IMSA", 2026, "Daytona", "Race")
print(imsa)

imsa_results = imsa.results()
imsa_results[["position", "car_number", "car_class", "team", "laps_completed"]].head(10)

---

## Next steps

- [API Documentation](https://api.openwec.com/docs) — full endpoint reference
- [openwec.com](https://openwec.com) — live dashboard
- [GitHub](https://github.com/palomacdev/openwec) — source code
- [Request API key](https://openwec.com/api-keys) — for laps and analytics endpoints